# Stage 2: Classical Bank Scorecard (WOE + LR)

**Objective**: Build an industry-standard credit scorecard using Weight of Evidence (WOE) encoding and Logistic Regression.

**Pipeline**: WOE Binning → IV Filter → WOE Transform → VIF/Corr Filter → LR → Score Scale (300-850)

**Output**: Scorecard table, KS/AUC metrics, score distribution plots, saved model.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_palette('muted')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'results' / 'analysis_reports'
MODEL_DIR = PROJECT_ROOT / 'results' / 'models'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:     {DATA_DIR}')
print(f'Report dir:   {REPORT_DIR}')

In [ ]:
# Load processed data
train = pd.read_parquet(DATA_DIR / 'train.parquet')
val = pd.read_parquet(DATA_DIR / 'val.parquet')
test = pd.read_parquet(DATA_DIR / 'test.parquet')

print(f'Train: {train.shape[0]:,} rows, {train.shape[1]} cols, bad_rate={train["is_bad"].mean():.2%}')
print(f'Val:   {val.shape[0]:,} rows, {val.shape[1]} cols, bad_rate={val["is_bad"].mean():.2%}')
print(f'Test:  {test.shape[0]:,} rows, {test.shape[1]} cols, bad_rate={test["is_bad"].mean():.2%}')

---
# Part 1: Train Scorecard
---

In [ ]:
from src.models.scorecard import Scorecard

sc = Scorecard(
    base_score=600,
    base_odds=50,
    pdo=20,
    iv_threshold=0.02,
    vif_threshold=5.0,
    corr_threshold=0.7,
    max_bins=6,
    min_bin_size=0.02,
    lr_C=1.0,
)

sc.fit(train, val, target='is_bad')

---
# Part 2: Model Evaluation
---

In [ ]:
from src.evaluation.metrics import scorecard_report, ks_score, gini_score, auc_score, pr_auc

# Predict on all sets
y_train_true = train['is_bad'].values
y_val_true = val['is_bad'].values
y_test_true = test['is_bad'].values

y_train_pred = sc.predict_proba(train)
y_val_pred = sc.predict_proba(val)
y_test_pred = sc.predict_proba(test)

y_train_score = sc.predict_score(train)
y_val_score = sc.predict_score(val)
y_test_score = sc.predict_score(test)

# Reports
report_train = scorecard_report(y_train_true, y_train_pred, y_train_score, 'Train')
report_val = scorecard_report(y_val_true, y_val_pred, y_val_score, 'Val')
report_test = scorecard_report(y_test_true, y_test_pred, y_test_score, 'Test')

print('=' * 70)
print(f'{"STAGE 2 — SCORECARD RESULTS":^70}')
print('=' * 70)
for r in [report_train, report_val, report_test]:
    print(f'\n--- {r["label"]} ---')
    print(f'  AUC:     {r["AUC"]:.4f}')
    print(f'  KS:      {r["KS"]:.4f}')
    print(f'  Gini:    {r["Gini"]:.4f}')
    print(f'  PR-AUC:  {r["PR_AUC"]:.4f}')
    print(f'  Default Rate: {r["default_rate"]:.2%}')
    print(f'  Score Bands:')
    for band, info in r.get('score_bands', {}).items():
        print(f'    {band}: n={info["count"]:,}  default_rate={info["default_rate"]:.2%}')

---
# Part 3: ROC Curves
---

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(10, 8))

for name, y_true, y_pred in [
    ('Train', y_train_true, y_train_pred),
    ('Val', y_val_true, y_val_pred),
    ('Test', y_test_true, y_test_pred),
]:
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — WOE Scorecard')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / '11_scorecard_roc.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 4: Score Distribution
---

In [ ]:
from src.evaluation.calibration import psi, score_distribution_stats

psi_train_test = psi(y_train_score, y_test_score, bins=10)
psi_train_val = psi(y_train_score, y_val_score, bins=10)

print(f'PSI (Train vs Test): {psi_train_test:.4f}')
print(f'PSI (Train vs Val):  {psi_train_val:.4f}')

stats = score_distribution_stats(y_train_score, y_test_score, y_val_score)
print(f'\nScore Distribution Stats:')
print(stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Score histogram
ax = axes[0]
for scores, name, color in [
    (y_train_score, 'Train', '#2ca02c'),
    (y_val_score, 'Val', '#ff7f0e'),
    (y_test_score, 'Test', '#d62728'),
]:
    ax.hist(scores, bins=50, alpha=0.4, density=True, label=name, color=color)
ax.set_xlabel('Credit Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Dataset')
ax.legend()

# Score by good/bad
ax = axes[1]
for mask, name, color in [
    (y_test_true == 0, 'Good (Fully Paid)', '#2ca02c'),
    (y_test_true == 1, 'Bad (Charged Off)', '#d62728'),
]:
    ax.hist(y_test_score[mask], bins=50, alpha=0.5, density=True, label=name, color=color)
ax.set_xlabel('Credit Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution: Good vs Bad (Test Set)')
ax.legend()

plt.tight_layout()
plt.savefig(REPORT_DIR / '12_scorecard_score_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 5: Default Rate by Score Band
---

In [ ]:
from src.evaluation.calibration import default_rate_by_score_band

band_df = default_rate_by_score_band(y_test_score, y_test_true, bins=8)
print('Default Rate by Score Band (Test):')
print(band_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    [f'{r["score_low"]}-{r["score_high"]}' for _, r in band_df.iterrows()],
    band_df['default_rate'],
    color='steelblue',
    edgecolor='white'
)
ax.set_xlabel('Score Band')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by Score Band — Test Set')
ax.tick_params(axis='x', rotation=45)
for bar, rate, count in zip(bars, band_df['default_rate'], band_df['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{rate:.1f}%\n(n={count:,})', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(REPORT_DIR / '13_scorecard_default_rate_bands.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 6: Feature Importance
---

In [ ]:
importance_df = sc.get_feature_importance()
top_features = importance_df.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d62728' if c < 0 else '#2ca02c' for c in reversed(top_features['coefficient'])]
ax.barh(range(len(top_features)), reversed(top_features['abs_coef']), color=colors, edgecolor='white')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(reversed(top_features['feature']))
ax.invert_yaxis()
ax.set_xlabel('|Coefficient|')
ax.set_title('Top Features by Absolute Coefficient')

for i, (_, row) in enumerate(reversed(list(top_features.iterrows()))):
    direction = '↑ risk' if row['coefficient'] > 0 else '↓ risk'
    ax.text(row['abs_coef'] + 0.01, i, f"{row['abs_coef']:.3f} ({direction})",
            va='center', fontsize=9)

plt.tight_layout()
plt.savefig(REPORT_DIR / '14_scorecard_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFull importance for {len(importance_df)} features:')
print(importance_df.to_string())

---
# Part 7: Scorecard Table Excerpt
---

In [ ]:
# Show top 3 features with full bin details
top3 = importance_df.head(3)['feature'].tolist()
table = sc.scorecard_table_
excerpt = table[table['feature'].isin(top3)]
print('=' * 120)
print(f'{"SCORECARD TABLE — Top 3 Features":^120}')
print('=' * 120)
print(excerpt[['feature', 'bin', 'count', 'event_rate', 'woe', 'coefficient', 'points']].to_string(index=False))
print('\n... (full scorecard has', len(table), 'rows)')

---
# Part 8: Save Model
---

In [ ]:
model_path = MODEL_DIR / 'scorecard_stage2.pkl'
sc.save(model_path)
print(f'Scorecard saved to: {model_path}')
print(f'File size: {model_path.stat().st_size / 1024:.1f} KB')

# Save scorecard table as CSV for inspection
table_path = REPORT_DIR / 'scorecard_table.csv'
sc.scorecard_table_.to_csv(table_path, index=False)
print(f'Scorecard table saved to: {table_path}')

---
# Part 9: Benchmark Comparison
---

In [ ]:
print('=' * 70)
print(f'{"BENCHMARK COMPARISON":^70}')
print('=' * 70)
print(f'\n{"Model":<30} {"Test AUC":>10} {"Test KS":>10}')
print('-' * 50)
print(f'{"Stage 1 — Raw LR":<30} {"0.7140":>10} {"---":>10}')
print(f'{"Stage 2 — WOE Scorecard":<30} {report_test["AUC"]:>10.4f} {report_test["KS"]:>10.4f}')
print()

auc_improvement = (report_test['AUC'] - 0.7140) / 0.7140 * 100
print(f'AUC improvement: {auc_improvement:+.1f}% vs Stage 1 baseline')

---
## Stage 2 Summary

- **Method**: WOE binning → IV filter → VIF/corr filter → Logistic Regression → Score scale
- **Features used**: {len(sc.final_features_)} out of original 89
- **Key metrics**: AUC={report_test['AUC']}, KS={report_test['KS']}, Gini={report_test['Gini']}
- **PSI (Train→Test)**: {psi_train_test:.4f}
- **Model saved**: `results/models/scorecard_stage2.pkl`
- **Scorecard table**: `results/analysis_reports/scorecard_table.csv`